# NB00 · 机器基线档案

| | |
|---|---|
| **目标** | 把这台机器的算力变成一份可对比、可复查的基线档案，决定「什么活本地干、什么活上云」 |
| **前置** | day0 安装完成，`check_env.sh` 全绿 |
| **预计耗时** | 20 分钟 |
| **产出物** | `results/NB00.json`（机器档案） |
| **通过标准** | 全部 cell 跑通且结果落盘 |

规则：从上到下顺序执行；每个 ✅ 检查点必须核对；最后的复盘必须填写并 commit。


In [1]:
import platform, shutil, subprocess, sys
import nbutils

profile = {"python": sys.version.split()[0], "platform": platform.platform()}

# CPU / RAM / disk
try:
    with open("/proc/meminfo") as f:
        kb = int(f.readline().split()[1])
    profile["ram_gb"] = round(kb / 1e6, 1)
except FileNotFoundError:
    profile["ram_gb"] = None  # macOS: sysctl hw.memsize
profile["disk_free_gb"] = round(shutil.disk_usage("/").free / 1e9, 1)
profile["cpu_count"] = __import__("os").cpu_count()
profile

{'python': '3.12.14',
 'platform': 'macOS-26.4.1-arm64-arm-64bit',
 'ram_gb': None,
 'disk_free_gb': 787.1,
 'cpu_count': 18}

In [2]:
# GPU 探测 + 矩阵乘吞吐（这是你之后估算训练时长的换算系数）
import time, torch

if torch.cuda.is_available():
    dev = torch.device("cuda"); profile["gpu"] = torch.cuda.get_device_name(0)
    profile["vram_gb"] = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    dev = torch.device("mps"); profile["gpu"] = "Apple MPS"
else:
    dev = torch.device("cpu"); profile["gpu"] = None

n = 2048
a, b = torch.randn(n, n, device=dev), torch.randn(n, n, device=dev)
for _ in range(3): a @ b
if dev.type == "cuda": torch.cuda.synchronize()
t0 = time.time(); iters = 20
for _ in range(iters): a @ b
if dev.type == "cuda": torch.cuda.synchronize()
dt = (time.time() - t0) / iters
profile["matmul_gflops"] = round(2 * n**3 / dt / 1e9)
print(f"device={profile['gpu'] or 'CPU'}  matmul={profile['matmul_gflops']} GFLOP/s")

ModuleNotFoundError: No module named 'torch'

In [ ]:
# MuJoCo 物理步进速度（决定 eval / 数据生成的本地吞吐）
import time, mujoco

XML = '<mujoco><worldbody><geom type="plane" size="1 1 0.1"/><body pos="0 0 1"><freejoint/><geom type="box" size="0.05 0.05 0.05"/></body></worldbody></mujoco>'
m = mujoco.MjModel.from_xml_string(XML); d = mujoco.MjData(m)
t0 = time.time(); steps = 20000
for _ in range(steps): mujoco.mj_step(m, d)
profile["mujoco_steps_per_s"] = round(steps / (time.time() - t0))
print(f"{profile['mujoco_steps_per_s']:,} steps/s")

In [ ]:
nbutils.log_result("NB00", profile)
profile

## 分析

参考换算（供对照，不是标准答案）：

| 机器 | matmul GFLOP/s | pusht 上训 Diffusion Policy 10 万步 |
|---|---|---|
| RTX 4090 | ~150,000+ | ~2–4 小时 |
| RTX 3060 (12GB) | ~30,000 | ~8–12 小时 |
| Apple M 系列 (MPS) | ~5,000–15,000 | 不建议，上云 |
| CPU | <2,000 | 不可行 |

**回答三个问题（写在下面）**：
1. 这台机器的瓶颈是什么（算力 / 显存 / 磁盘 / 都不是）？
2. NB02/04/06/08 的训练放本地还是云？预算多少？
3. eval 和数据处理放哪？（提示：几乎永远是本地）


## 复盘（必填，不填不算完成这本 notebook）

> 复盘写在这里并 commit。允许粗糙，禁止事后美化。

- **预期 vs 实际**：
- **最大的一个意外**：
- **卡最久的一步和根因**：
- **用一句话向非技术人解释本次学到的东西**：
- **进入下一本之前要做的一个动作**：
